# 1. Imports

In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# 2. Connection

In [2]:
load_vars = load_dotenv("../src/.env")
if load_vars:
    print("Variáveis de ambiente carregadas com sucesso")
else:
    print("Arquivo .env não encontrado!")

Variáveis de ambiente carregadas com sucesso


In [3]:
connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=tcp:127.0.0.1,1433;"
    f"DATABASE={os.environ['DB_NAME']};"
    f"UID={os.environ['DB_USER']};"
    f"PWD={os.environ['DB_PASSWORD']};"
    "TrustServerCertificate=yes;"
    "Encrypt=no;"
)

In [4]:
connection_url = f"mssql+pyodbc:///?odbc_connect={quote_plus(connection_string)}"

In [5]:
engine = create_engine(connection_url)

# 3. Pegando todas tabelas

In [6]:
tables = [
    'dados_receitas',
    'fila_batch_ids',
    'fila_paradas',
    'ihms',
    'linhas_producao',
    'logs_registradores',
    'notificacoes',
    'ordens_servico',
    'paradas',
    'parametros',
    'receitas',
    'registradores',
    'sistemas'
]

In [7]:
dicionario_dfs = dict()
for table in tables:
    df_apoio = pd.read_sql(f"SELECT * FROM {table}", engine)
    if len(df_apoio) > 0:
        dicionario_dfs[table] = df_apoio
    del df_apoio

In [8]:
# for key in dicionario_dfs.keys():
#     print(key)
#     print(dicionario_dfs[key].head())
#     print('------')
    

# 4. Tratando informação

In [9]:
df_interesse = dicionario_dfs['logs_registradores'].merge(dicionario_dfs['ihms'].rename(columns={'id':'id_ihm'})[['id_ihm', 'nome_maquina']], how='left', on='id_ihm')

In [10]:
df_interesse = df_interesse.merge(dicionario_dfs['registradores'].rename(columns={'id':'id_registrador'})[['id_registrador', 'descricao']], how='left', on='id_registrador')

In [11]:
df_interesse = df_interesse[['nome_maquina', 'descricao', 'datahora', 'valor_bruto']]

In [12]:
df_interesse = df_interesse.pivot_table(index=['nome_maquina','datahora'], columns='descricao', values='valor_bruto', aggfunc='first').reset_index()

In [13]:
df_interesse = df_interesse.sort_values('datahora')
df_interesse.reset_index(drop=True, inplace=True)

In [14]:
df_interesse.columns

Index(['nome_maquina', 'datahora', 'engenheiro', 'manutentor', 'motivo_parada',
       'operador', 'produzido', 'reprovado', 'status_maquina',
       'total_produzido'],
      dtype='object', name='descricao')

In [15]:
depara_status_maquina = {
    '0': 'Parada',
    '1': 'Passar Padrão',
    '49': 'Produzindo',
    '4': 'Limpeza'
}

In [16]:
df_interesse['status_maquina'] = df_interesse['status_maquina'].map(depara_status_maquina)

In [17]:
df_interesse = df_interesse[df_interesse['nome_maquina']=='MAQ1']

In [18]:
relatorio = "Iniciando relatório..."
colunas_interesse = ['engenheiro', 'manutentor', 'motivo_parada',
       'operador', 'produzido', 'reprovado', 'status_maquina',
       'total_produzido']
dicionario_controle = dict()
for i, row in df_interesse.iterrows():
    relatorio += f"\n\n**{row['datahora']}**" 
    if i > 0:
        relatorio += "\nMudanças de Estado"
    for col in colunas_interesse:
        if col not in dicionario_controle.keys():
            relatorio += f"\n{col} (Estado inicial): {row[col]}"
            dicionario_controle[col] = row[col]
        else:
            if dicionario_controle[col] != row[col]:
                relatorio += f"\n{col}: {row[col]}"
                dicionario_controle[col] = row[col]

In [19]:
print(relatorio)

Iniciando relatório...

**2025-11-25 19:58:57.623000**
engenheiro (Estado inicial): 0
manutentor (Estado inicial): 0
motivo_parada (Estado inicial): 49
operador (Estado inicial): 7
produzido (Estado inicial): 59
reprovado (Estado inicial): 13
status_maquina (Estado inicial): Produzindo
total_produzido (Estado inicial): 72

**2025-11-26 14:59:54.543000**
Mudanças de Estado
manutentor: 72
status_maquina: Parada

**2025-11-26 15:00:03.890000**
Mudanças de Estado
operador: 49
produzido: 7
reprovado: 59
total_produzido: 13

**2025-11-26 15:00:17.597000**
Mudanças de Estado
manutentor: 0
reprovado: 13
total_produzido: 72

**2025-11-26 15:00:30.840000**
Mudanças de Estado
status_maquina: Produzindo

**2025-11-26 15:00:51.633000**
Mudanças de Estado
manutentor: 72
operador: 7
reprovado: 59
status_maquina: Parada
total_produzido: 13

**2025-11-26 15:01:09.373000**
Mudanças de Estado
manutentor: 0
operador: 49
reprovado: 13
status_maquina: Produzindo
total_produzido: 72

**2025-11-26 15:01:15.89

In [20]:
df_interesse#.head()

descricao,nome_maquina,datahora,engenheiro,manutentor,motivo_parada,operador,produzido,reprovado,status_maquina,total_produzido
0,MAQ1,2025-11-25 19:58:57.623,0,0,49,7,59,13,Produzindo,72
4,MAQ1,2025-11-26 14:59:54.543,0,72,49,7,59,13,Parada,72
5,MAQ1,2025-11-26 15:00:03.890,0,72,49,49,7,59,Parada,13
6,MAQ1,2025-11-26 15:00:17.597,0,0,49,49,7,13,Parada,72
7,MAQ1,2025-11-26 15:00:30.840,0,0,49,49,7,13,Produzindo,72
...,...,...,...,...,...,...,...,...,...,...
114,MAQ1,2025-11-26 15:23:33.340,0,119,49,49,7,91,Parada,28
115,MAQ1,2025-11-26 15:24:04.293,0,0,49,49,7,28,Produzindo,119
117,MAQ1,2025-11-26 15:24:10.817,0,0,49,7,91,28,Produzindo,119
119,MAQ1,2025-11-26 15:24:24.300,0,119,49,7,7,91,Parada,28


In [21]:
df_interesse.shape

(63, 10)

In [22]:
def get_metrics_machine(df_timeline, machine='MAQ1'):
    
    first_register = df_timeline[df_timeline['datahora']==df_timeline['datahora'].min()]
    last_register = df_timeline[df_timeline['datahora']==df_timeline['datahora'].max()]
    status = last_register['status_maquina'].to_list()[0]
    operador = last_register['operador'].to_list()[0]
    manutentor = last_register['manutentor'].to_list()[0]
    produzido = last_register['produzido'].to_list()[0]
    reprovado = last_register['reprovado'].to_list()[0]
    total = last_register['total_produzido'].to_list()[0]
    
    # OEE = Disponibilidade * Performance * Qualidade
    
    # Disponibilidade = Tempo produzido / Tempo programado para produzir
    lista_produzido = []
    status_antigo = ""
    inicio = None
    fim = None
    for i, row in df_timeline.iterrows():
        if status_antigo != 'Produzindo' and row['status_maquina']=='Produzindo':
            inicio = row['datahora']
        elif (status_antigo == 'Produzindo' and row['status_maquina']!='Produzindo') or (status_antigo == 'Produzindo' and row['status_maquina']=='Produzindo' and row['datahora']==last_register['datahora'].to_list()[0]):
            fim = row['datahora']
        if inicio and fim:
            lista_produzido.append((inicio, fim))
            inicio = None
            fim = None            
        status_antigo = row['status_maquina']
    tempo_produzido = sum([y.total_seconds() for y in [x[1] - x[0] for x in lista_produzido]])
    tempo_programado = (last_register['datahora'].to_list()[0] - first_register['datahora'].to_list()[0]).total_seconds()
    disponibilidade = tempo_produzido / tempo_programado
        
    # Performance = Produção Real / Produção Teórica
    meta = (tempo_programado // 1) # Considerando que a cada 1 s é feito uma peça
    performance = int(total) / meta
    
    # Qualidade = Peça Boas / Total de peças
    qualidade = int(produzido) / int(total)
    
    oee = disponibilidade * performance * qualidade
    # OEE, Qualidade, Eficiencia, Meta, Acumulado, Operador, Manutentor, Status
    
    
    return {
        'status': status,
        'oee': round(100 * oee, 2),
        'eficiencia': performance, # eficiencia é isso mesmo?
        'qualidade': round(100 * qualidade, 2),
        'meta': meta,
        'produzido': produzido,
        'reprovado': reprovado,
        'total': total,
        'operador': operador,
        'manutentor': manutentor
    }
    

In [23]:
get_metrics_machine(df_interesse)

{'status': 'Parada',
 'oee': 0.01,
 'eficiencia': 0.0004002801961372961,
 'qualidade': 25.0,
 'meta': 69951.0,
 'produzido': '7',
 'reprovado': '91',
 'total': '28',
 'operador': '49',
 'manutentor': '119'}